In [0]:
%run ../utils

In [0]:
# Create leads_processed_custom from bronze leads_raw
# Calls extract_custom with dry_run=False to actually WRITE the child table
import time
start = time.time()

print(f"{'='*80}")
print(f"📥 leads_raw → leads_processed_custom")
print(f"{'='*80}")

# Check CDC watermark for the custom table
watermark = get_silver_watermark('leads_processed_custom')
if watermark:
    print(f"  CDC MODE: reading changes after {watermark}")
else:
    print(f"  FULL LOAD: first run")

# Step 1: Parse bronze table
df_flat = parse_bronze_table('leads_raw', watermark=watermark)
if df_flat is None:
    print(f"  ✅ CDC: No new rows since {watermark}")
    # Print existing table details so output matches first run
    all_tables = spark.catalog.listTables("crm_ingestion.silver")
    related = [t for t in all_tables if t.name == 'leads_processed_custom' or t.name.startswith('leads_processed_custom_')]
    print(f"\n  📊 Existing silver tables ({len(related)}):")
    for t in related:
        tdf = spark.table(f"crm_ingestion.silver.{t.name}")
        cnt = tdf.count()
        cols = tdf.columns
        print(f"     📦 {t.name} ({len(cols)} cols, {cnt:,} rows): {cols}")
    print(f"\n  📋 LINEAGE TREE:")
    print(f"  # * 1_1_leads_processed_custom")
    for t in related:
        if t.name != 'leads_processed_custom':
            print(f"  # * ------1_1_{t.name}")
    print(f"{'='*80}")
else:
    # Step 2: Flatten structs (keep custom struct as-is for extraction)
    df_flat = flatten_structs(df_flat, skip_custom=True)

    # Step 3: Find PK and build carry columns
    pk = find_pk(df_flat)
    carry = list(dict.fromkeys(["bronze_insert_date", "bronze_updated_at"] + ([pk] if pk else [])))

    # Step 4: Extract custom with dry_run=False to actually write the table
    # CDC FIX: propagate the watermark to write_silver (module global via %run)
    # so the write is delete+append matching the filtered read above —
    # previously this full-overwrote the table with ONLY the new rows.
    _CDC_WATERMARK = watermark
    has_custom = "custom" in df_flat.columns and isinstance(df_flat.schema["custom"].dataType, StructType)
    has_custom_cf = any(c.startswith("custom_cf_") for c in df_flat.columns)

    if has_custom or has_custom_cf:
        df_flat = extract_custom(df_flat, 'leads_processed', carry, "1_1", parent_pk=pk, dry_run=False)
        print(f"\n  📋 LINEAGE TREE:")
        for num, depth, name in _lineage_tree:
            print(f"  # * {'---' * depth}{num}_{name}")
    else:
        print("  ⚠️ No custom fields found")

    # Reset CDC watermark (mirrors process_table hygiene)
    _CDC_WATERMARK = None

    # Clean up temp tables
    for temp in _TEMP_TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {temp}")
    _TEMP_TABLES.clear()

    print(f"\n  ⏱️ Completed in {time.time() - start:.1f}s")
    print(f"{'='*80}")

In [0]:
display(
    spark.table('crm_ingestion.silver.leads_processed_custom')
    .filter("Contract_Start_Date = '2025-02-10'")
    .filter("Contract_End_Date = '2026-11-10'")
    .filter("leads_processed_id = 'lead_nESTKYYJbOzXiMeZcu3meMMEBLworCUfPMW4H95EkcH'")

    )

In [0]:
%sql
-- Latest custom-field snapshot per lead (table keeps ~6 bronze versions per lead,
-- so always dedupe with QUALIFY or you get 6x rows per lead)
SELECT leads_processed_id,
       Contract_Start_Date,
       Contract_End_Date,
       Contracted_Value,
       Tier,
       Subscription,
       ENGAGEMENT_PATTERN,
       DAYS_SINCE_LAST_LOGIN,
       HEALTH_BAND,
       bronze_insert_date
FROM crm_ingestion.silver.leads_processed_custom
QUALIFY ROW_NUMBER() OVER (PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) = 1
ORDER BY bronze_insert_date   DESC